# B3 · Custom features inside the pipeline, and saving the model for production

**Type**: Implement with scikit-learn · **Level**: ●●○ independent · **Time**: about 40 minutes · **core path ★**

## Objective

Add computed features (family size, logarithmic fare) to the pipeline with `FunctionTransformer`, save
the whole pipeline with `joblib`, reload it and use it on a **single request** the way a production service would.

## Dataset

`ds.titanic()`: stratified 75/25 split as in the previous exercises. The function `add_features(df)` (to write) receives a
DataFrame and returns a copy with three more columns.

## Educational purpose

Understand that the **feature engineering code is part of the model**: if you run it outside the pipeline, in production you have to
rewrite it (and get it wrong). Inside the pipeline it travels with the model, and saving with `joblib` preserves it.

## Task

1. Write `add_features(df)`: it returns a copy of `df` with `family_size = siblings_spouses + parents_children + 1`, `alone = (family_size == 1).astype(int)` and `log_fare = np.log1p(fare)`. Don't modify `df` in place.
2. Build `pipe`: `FunctionTransformer(add_features)` as the first step, then a `ColumnTransformer` that handles `NUM + ["family_size", "log_fare"]` (median + scaler), `CAT` (mode + one-hot), and passes through `pclass` and `alone` (`"passthrough"`), then the logistic.
3. Train on the training set and save `test_auc`. Check that `names` (`get_feature_names_out` of the preprocessor) contains `num__family_size`.
4. Save the pipeline with `joblib.dump(pipe, path)` (`path` is a temporary file), reload it in `loaded` and compute `single_p`: the survival probability for **one** request (a dict turned into a one-row DataFrame, `request`).
5. Check that `single_p` matches the probability that the original pipeline gives for the same row and save `identical`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = ds.titanic()
X, y = df.drop(columns="survived"), df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
NUM = ["age", "siblings_spouses", "parents_children", "fare"]
CAT = ["sex", "port", "deck", "title"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)

import tempfile
from pathlib import Path

import joblib
from sklearn.preprocessing import FunctionTransformer

path = Path(tempfile.mkdtemp()) / "titanic_model.joblib"
request = pd.DataFrame([{"pclass": 3, "sex": "male", "age": 22.0, "siblings_spouses": 1, "parents_children": 0,
                         "fare": 7.25, "port": "S", "deck": None, "title": "Mr"}])

## Your work

In [ ]:
def add_features(df):
    ...

In [ ]:
pipe = make_pipeline(
    FunctionTransformer(add_features),
    ColumnTransformer([...], sparse_threshold=0),
    LogisticRegression(max_iter=2000),
)
pipe.fit(X_tr, y_tr)
test_auc = ...
names = list(pipe[1].get_feature_names_out())

In [ ]:
joblib.dump(pipe, path)
loaded = ...
single_p = ...
identical = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`add_features`: `d = df.copy(); d["family_size"] = d.siblings_spouses + d.parents_children + 1; ...; return d`. The ColumnTransformer: `("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM + ["family_size", "log_fare"])`, `("cat", ..., CAT)`, `("pass", "passthrough", ["pclass", "alone"])`. Loading: `joblib.load(path)`.'):
    small = pd.DataFrame({"siblings_spouses": [1, 0], "parents_children": [2, 0], "fare": [10.0, 0.0]})
    out = add_features(small)
    check(list(out["family_size"]) == [4, 1] and list(out["alone"]) == [0, 1] and close(out["log_fare"], np.log1p([10.0, 0.0])),
          "add_features computes the three columns", "family_size = siblings + parents + 1; alone = family_size == 1; log_fare = log1p(fare)")
    check("family_size" not in small.columns, "the input is not modified in place", "use df.copy()")
    check(0.80 < test_auc < 0.93, "reasonable test AUC", "test_auc = AUC on the test set of the trained pipeline")
    check("num__family_size" in names and "num__log_fare" in names and any(n.endswith("alone") for n in names),
          "the new features reach the model", "include family_size and log_fare among the numerics and alone in passthrough")
    check(path.exists() and path.stat().st_size > 0, "the joblib file exists", "joblib.dump(pipe, path)")
    check(0 <= float(single_p) <= 1, "single_p is a probability", "predict_proba(request)[0, 1]")
    check(bool(identical) and close(single_p, pipe.predict_proba(request)[0, 1]), "the reloaded pipeline gives the same result as the original",
          "compare with pipe.predict_proba(request)")
done()

---
The commented solution is in `solutions/B_with_sklearn/B3_custom_features_and_saving.ipynb`. Open it after you pass the check, or after you have really tried.